# 10. LightGBM: внешние факторы и соседи по графам (Kaggle)

Запускается на Kaggle. Код берётся из GitHub (`Kinggl3/vkr-transit`), данные — датасет `vkr-transit-data`.

Что считается (docs/PROTOCOL.md):

| модель | соседи | гипотеза |
|---|---|---|
| `lgbm` | нет | H1: внешние факторы против `norm_level` |
| `lgbm_line`, `lgbm_dist`, `lgbm_od_in`, `lgbm_od_dyn` | один граф | H2, H3: дают ли соседи выигрыш и какой граф лучше |
| `lgbm_all` | все четыре | H2 |
| `lgbm_wactual` | нет, погода — факт | H5: цена прогноза погоды (только val) |

**Настройки сессии:** Accelerator — None (CPU), Internet — On, Persistence — Files.
Запуск: Save Version → Save & Run All. Результаты — во вкладке Output: `results_csv.zip` (метрики, журнал,
важность признаков) — скачать и распаковать в `results/` репозитория.

In [ ]:
import os, glob, subprocess, sys, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")))
print("events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
!nproc; free -g | head -2; python -c "import lightgbm; print('lightgbm', lightgbm.__version__)"

## 1. Бейзлайны (эталон для сравнения, ~1 мин)

In [ ]:
!cd {REPO} && python src/baselines.py

## 2. LightGBM

Основные модели (`lgbm`, `lgbm_all`) прогнозируют все точки val и test. Варианты для сравнения графов и погоды —
только общий подсет (каждая 7-я точка, протокол, раздел 2): так всё укладывается в одну сессию (~6–7 ч на CPU).
Каждый запуск печатает время; уже посчитанные модели при повторном запуске пропускаются.

In [ ]:
RUNS = [   # имя, графы соседей, погода на val/test, выборки, шаг точек прогноза (1 — все, 7 — общий подсет)
    ("lgbm",         [],                                   "forecast", "val test", 1),
    ("lgbm_all",     ["line", "dist", "od_in", "od_dyn"],  "forecast", "val test", 1),
    ("lgbm_line",    ["line"],                             "forecast", "val test", 7),
    ("lgbm_dist",    ["dist"],                             "forecast", "val test", 7),
    ("lgbm_od_in",   ["od_in"],                            "forecast", "val test", 7),
    ("lgbm_od_dyn",  ["od_dyn"],                           "forecast", "val test", 7),
    ("lgbm_wactual", [],                                   "actual",   "val",      7),
]
for name, graphs, weather, splits, ps in RUNS:
    if os.path.exists(f"/kaggle/working/results/preds/{name}__val.npz"):
        print(f"{name}: уже посчитан, пропускаю"); continue
    t = time.time()
    g = f"--graphs {' '.join(graphs)}" if graphs else ""
    !cd {REPO} && python src/train_lgbm.py --name {name} {g} --weather {weather} --splits {splits} --pred-stride {ps}
    print(f"=== {name}: {(time.time() - t) / 60:.0f} мин\n")

## 3. Оценка

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

## 4. Сравнение по горизонтам и важность признаков

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 20)
R = "/kaggle/working/results"
m = pd.read_csv(f"{R}/metrics_val.csv")
C = m[m.subset == "общий подсет"]                      # здесь сравнимы все модели
hz = C[C.slice.str.startswith("горизонт")].pivot_table(index="model", columns="slice", values="wape")
hz = hz[["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]]
base = hz.loc["lgbm"]
print("WAPE на val (общий подсет) по горизонтам:"); display(hz.round(4).sort_values("горизонт h1"))
print("Относительное изменение к lgbm без соседей, %:"); display(((hz / base - 1) * 100).round(1).sort_values("горизонт h1"))
sl = C[C.slice.str.match("день:|событие|часы:")].pivot_table(index="slice", columns="model", values="wape")
display(sl.round(3))
for n in ["lgbm", "lgbm_all"]:
    p = f"{R}/fi_{n}.csv"
    if os.path.exists(p):
        print(f"Важность признаков {n} (доля gain, топ-15):"); display(pd.read_csv(p).head(15).round(3))

## 5. Архив результатов для репозитория

In [ ]:
import shutil, zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))
print("Прогнозы (большие, остаются в Output):", len(glob.glob(f"{R}/preds/*.npz")), "файлов")